In [1]:
!pip install transformers sentencepiece scikit-learn

In [9]:
import os
import torch
import warnings
import logging
import numpy as np
import pandas as pd
from transformers import (
    pipeline,
    AutoTokenizer, AutoModelForSeq2SeqLM,
    BertTokenizer, BertForSequenceClassification,
    AutoModelForSequenceClassification
)
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from torch.utils.data import Dataset, DataLoader
from google.colab import drive
import time
from tqdm import tqdm

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(device)}")

Using GPU: NVIDIA A100-SXM4-40GB


In [4]:
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger().setLevel(logging.ERROR)

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

drive.mount('/content/drive', force_remount=True)

drive_path = '/content/drive/MyDrive/Event Log/2025-7-28 申毅实习/策略研究/NLP 日度'
os.chdir(drive_path)

Mounted at /content/drive


In [5]:
device = 0 if torch.cuda.is_available() else -1
print("Device set to", "GPU" if device == 0 else "CPU")

Device set to GPU


In [6]:
# === Translator: facebook/nllb-200-3.3B ===
translator_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-3.3B")
translator_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-3.3B").to(device)

# === Classifier: FinBERT-Tone ===
classifier_tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
classifier_model = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone").to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
equity_reports = pd.read_csv(
    'equity_reports.csv',
    encoding='utf-8',
    parse_dates=['PUBLISH_DATE', 'UPDATE_TIME', 'UPDATE_DATE']
)

In [8]:
output_path = "sentiment_output.csv"

if os.path.exists(output_path):
    processed_ids = set(pd.read_csv(output_path)["REPORT_ID"])
    print(f"Resuming from progress log: {len(processed_ids)}")
else:
    processed_ids = set()
    pd.DataFrame(columns=[
        "REPORT_ID", "title_label", "title_score", "summary_label", "summary_score"
    ]).to_csv(output_path, index=False)
    print("Created new progress log.")

Resuming from progress log: 200


# Batch Loader

In [ ]:
class ReportDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe[~dataframe["REPORT_ID"].isin(processed_ids)].copy()
        self.df.fillna("", inplace=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "REPORT_ID": row["REPORT_ID"],
            "TITLE": row["TITLE"],
            "SUMMARY": row["SUMMARY"]
        }

batch_size = 16
dataset = ReportDataset(equity_reports)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def translate_batch(texts, max_length):
    inputs = translator_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        outputs = translator_model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            early_stopping=True
        )

    return translator_tokenizer.batch_decode(outputs, skip_special_tokens=True)

def classify_batch(texts):
    inputs = classifier_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        logits = classifier_model(**inputs).logits
        probs = torch.nn.functional.softmax(logits, dim=-1)
        scores, preds = torch.max(probs, dim=1)
        labels = [classifier_model.config.id2label[p.item()] for p in preds]

    return labels, scores.tolist()

In [ ]:
for batch in tqdm(dataloader, desc="Processing Batches"):
    ids = batch["REPORT_ID"]
    titles_cn = batch["TITLE"]
    summaries_cn = batch["SUMMARY"]

    # Translate
    titles_en = translate_batch(titles_cn, max_length=600)
    summaries_en = translate_batch(summaries_cn, max_length=1024)

    # Classify
    title_labels, title_scores = classify_batch(titles_en)
    summary_labels, summary_scores = classify_batch(summaries_en)

    # Build and save DataFrame immediately
    batch_results = pd.DataFrame({
        "REPORT_ID": ids,
        "title_label": title_labels,
        "title_score": title_scores,
        "summary_label": summary_labels,
        "summary_score": summary_scores
    })

    batch_results.to_csv(output_path, mode="a", header=False, index=False)
    print(f"Saved batch of {len(ids)} rows to {output_path}")
